# Customer Churn Prediction — Week 1: Exploratory Data Analysis

**Goal:** Merge 6 data sources, understand churn patterns, and generate business insights.

**Dataset:** IBM Telco Customer Churn (Extended — 6 Excel files)  
**Total customers:** 7,043  
**Target:** `Churn Label` — whether a customer left (Yes/No)


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Libraries loaded!')

## 2. Load and merge all 6 data files

In [ ]:
main         = pd.read_excel('../data/Telco_customer_churn.xlsx')
demographics = pd.read_excel('../data/Telco_customer_churn_demographics.xlsx')
services     = pd.read_excel('../data/Telco_customer_churn_services.xlsx')
status       = pd.read_excel('../data/Telco_customer_churn_status.xlsx')

# Standardize join key
for frame in [demographics, services, status]:
    frame.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

# Merge
df = main.copy()
df = df.merge(demographics[['CustomerID', 'Age', 'Married', 'Number of Dependents']], on='CustomerID', how='left')
df = df.merge(services[['CustomerID', 'Offer', 'Internet Type', 'Avg Monthly GB Download',
                          'Number of Referrals', 'Total Revenue']], on='CustomerID', how='left')
df = df.merge(status[['CustomerID', 'Satisfaction Score', 'Churn Category']], on='CustomerID', how='left')

# Fix Total Charges dtype
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce').fillna(0)

# Binary target
df['Churn_binary'] = (df['Churn Label'] == 'Yes').astype(int)

print(f'Merged dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

## 3. Data quality check

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).round(1)
missing = missing[missing > 0].sort_values(ascending=False)
print('Missing values (%):')
print(missing if not missing.empty else 'None found!')
print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 4. Churn rate overview

In [ ]:
churn_counts = df['Churn Label'].value_counts()
churn_pct    = df['Churn Label'].value_counts(normalize=True) * 100

print(f"Retained : {churn_counts['No']:,}  ({churn_pct['No']:.1f}%)")
print(f"Churned  : {churn_counts['Yes']:,}  ({churn_pct['Yes']:.1f}%)")

fig = px.pie(values=churn_counts.values, names=churn_counts.index,
             title='Customer Churn Distribution',
             color_discrete_map={'Yes': '#E74C3C', 'No': '#2ECC71'})
fig.show()

## 5. Churn by contract type

In [ ]:
contract_churn = (df.groupby('Contract')['Churn_binary']
                    .mean()
                    .mul(100).round(1)
                    .reset_index()
                    .rename(columns={'Churn_binary': 'Churn Rate %'})
                    .sort_values('Churn Rate %', ascending=False))

fig = px.bar(contract_churn, x='Contract', y='Churn Rate %',
             title='Churn Rate by Contract Type',
             color='Churn Rate %', color_continuous_scale='Reds', text='Churn Rate %')
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

## 6. Why are customers churning?

In [ ]:
# Top churn reasons
reasons = (df[df['Churn Label'] == 'Yes']['Churn Reason']
             .value_counts().head(10).reset_index())
reasons.columns = ['Reason', 'Count']

fig = px.bar(reasons, x='Count', y='Reason', orientation='h',
             title='Top 10 Reasons Customers Churned',
             color='Count', color_continuous_scale='Reds')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Churn by category
churn_cat = (df[df['Churn Label'] == 'Yes']['Churn Category']
               .value_counts().reset_index())
churn_cat.columns = ['Category', 'Count']

fig = px.pie(churn_cat, values='Count', names='Category',
             title='Churn by Category (e.g. Competitor, Dissatisfaction)')
fig.show()

## 7. Satisfaction score vs churn

In [ ]:
sat = df.groupby(['Satisfaction Score', 'Churn Label']).size().reset_index(name='Count')

fig = px.bar(sat, x='Satisfaction Score', y='Count', color='Churn Label',
             barmode='group', title='Satisfaction Score Distribution by Churn',
             color_discrete_map={'Yes': '#E74C3C', 'No': '#2ECC71'})
fig.show()

## 8. Tenure and Monthly Charges vs churn

In [ ]:
fig = px.scatter(df, x='Tenure Months', y='Monthly Charges', color='Churn Label',
                 color_discrete_map={'Yes': '#E74C3C', 'No': '#2ECC71'},
                 title='Tenure vs Monthly Charges — Colored by Churn',
                 opacity=0.4,
                 labels={'Tenure Months': 'Tenure (months)',
                         'Monthly Charges': 'Monthly Charges ($)'})
fig.show()

## 9. Age distribution by churn

In [ ]:
fig = px.histogram(df, x='Age', color='Churn Label', barmode='overlay',
                   title='Age Distribution by Churn',
                   color_discrete_map={'Yes': '#E74C3C', 'No': '#2ECC71'},
                   opacity=0.7, nbins=30)
fig.show()

## 10. Correlation heatmap

In [ ]:
num_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges',
            'Age', 'Satisfaction Score', 'CLTV', 'Churn Score', 'Churn_binary']

corr = df[num_cols].corr()

fig = px.imshow(corr, title='Correlation Heatmap',
                color_continuous_scale='RdBu_r',
                text_auto='.2f', aspect='auto')
fig.show()

## 11. Key findings

Fill these in after running all cells above:

**Finding 1 — Overall churn rate:**  
> Write the % here and note whether it's a class imbalance problem.

**Finding 2 — Contract type:**  
> Which contract type churns most? Why does that make business sense?

**Finding 3 — Top churn reason:**  
> What is the #1 reason customers left?

**Finding 4 — Satisfaction score:**  
> Which scores are most associated with churn?

**Finding 5 — Tenure pattern:**  
> Do newer or longer-tenure customers churn more?

**Finding 6 — Top features for modeling:**  
> List the top 5 features you expect to matter most based on this EDA.

In [ ]:
# Save merged dataset for Week 2
df.to_csv('../data/telco_churn_merged.csv', index=False)
print(f'Saved: telco_churn_merged.csv — {df.shape[0]:,} rows x {df.shape[1]} columns')